In [1]:
import torch
torch.linspace(2,40,39)

tensor([ 2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12., 13., 14., 15.,
        16., 17., 18., 19., 20., 21., 22., 23., 24., 25., 26., 27., 28., 29.,
        30., 31., 32., 33., 34., 35., 36., 37., 38., 39., 40.])

: 

In [1]:
from utils import *

In [2]:
seq_id = '5e81_1K'

In [3]:
# Get sequence for the specified seq_id
from rhofold.utils.alphabet import get_features, read_fas
import os
import json
from pathlib import Path

def get_stuff(seq_id):
    """
    Retrieve the RNA sequence for the given seq_id.
    
    Args:
        seq_id: The sequence identifier
        
    Returns:
        The RNA sequence as a string
    """
    # Try to find the sequence in different possible locations
    
    seq_file_path = Path(f"/dev/shm/RNA3D_DATA/seq/{seq_id}.seq")
    seq = read_fas(seq_file_path)
    
    # Option 3: Check if it's in a PDB file
    from Bio.PDB import PDBParser
    pdb_path = Path(f"/dev/shm/RNA3D_DATA/pdb/{seq_id}.pdb")
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("structure", pdb_path)
    return seq, structure

In [4]:
pdb_path = "/dev/shm/RNA3D_DATA/pdb/5e81_1K.pdb"

In [5]:
from Bio.PDB import PDBParser
parser = PDBParser(PERMISSIVE=True)
len(list(parser.get_structure("structure",pdb_path).get_residues()))

72

In [6]:
next(parser.get_structure("structure",pdb_path).get_residues())["C1'"].get_coord()

array([  1.393, 265.784,  16.157], dtype=float32)

In [8]:
seq_id

'5e81_1K'

In [11]:
seq_l, structure = get_stuff(seq_id)

In [12]:
len(seq_l[0][1])

76

In [13]:
dir(structure)

['__annotations__',
 '__class__',
 '__class_getitem__',
 '__contains__',
 '__delattr__',
 '__delitem__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__len__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__orig_bases__',
 '__parameters__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__slots__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_generate_full_id',
 '_id',
 '_is_protocol',
 '_reset_full_id',
 'add',
 'atom_to_internal_coordinates',
 'center_of_mass',
 'child_dict',
 'child_list',
 'copy',
 'detach_child',
 'detach_parent',
 'full_id',
 'get_atoms',
 'get_chains',
 'get_full_id',
 'get_id',
 'get_iterator',
 'get_level',
 'get_list',
 'get_models',
 'get_parent',
 'get_residues',
 'has_id',
 'header',
 'id',
 'insert',
 'internal_to_atom_coordinates',
 'level',
 'parent',
 'set_par

In [14]:
len(structure)

1

In [15]:
seq = ""
for res in structure.get_residues():
    seq += res.get_resname()
print(seq)

GGGUCGUUAGCUCAGUGAGAGCAGUUGACUUUUAAUCAAUUGUCGCAGGUUCGAAUCCUGCACGACCCACCA


In [16]:
x = ""
pdb_ptr = 0
for index, res in enumerate(seq_l[0][1]):
    if res == seq[pdb_ptr]:
        pdb_ptr += 1
        x += res
    else:
        x += "?"
        print("skipped at index", index)
print(x)


skipped at index 16
skipped at index 18
skipped at index 19
skipped at index 45
GGGUCGUUAGCUCAGU?G??AGAGCAGUUGACUUUUAAUCAAUUG?UCGCAGGUUCGAAUCCUGCACGACCCACCA


: 

In [16]:
seq_l[0][1]

'GGGUCGUUAGCUCAGUUGGUAGAGCAGUUGACUUUUAAUCAAUUGGUCGCAGGUUCGAAUCCUGCACGACCCACCA'

In [20]:
'GGGUCGUUAGCUCAGUUGGUAGAGCAGUUGACUUUUAAUCAAUUGGUCGCAGGUUCGAAUCCUGCACGACCCACCA'

'GGGUCGUUAGCUCAGUUGGUAGAGCAGUUGACUUUUAAUCAAUUGGUCGCAGGUUCGAAUCCUGCACGACCCACCA'

In [17]:
seq_id

'5e81_1K'

In [2]:
import pickle
import torch
import os
from tqdm import tqdm

# Path to the pickle file
pickle_path = "/dev/shm/nvidia_evo2_full_embeddings.pkl"

# Check if the file exists
if os.path.exists(pickle_path):
    print(f"Loading embeddings from {pickle_path}...")
    with open(pickle_path, "rb") as f:
        evo2_embeddings = pickle.load(f)
    
    # Print some information about the loaded data
    print(f"Loaded embeddings for {len(evo2_embeddings)} sequences")
    
    # Display a sample key and the shape of its embedding if it's a tensor
    if evo2_embeddings:
        sample_key = next(iter(evo2_embeddings))
        sample_value = evo2_embeddings[sample_key]
        print(f"Sample key: {sample_key}")
        print(f"Sample value type: {type(sample_value)}")
        
        if hasattr(sample_value, 'shape'):
            print(f"Sample embedding shape: {sample_value.shape}")
else:
    print(f"Error: File {pickle_path} not found")

# Create the output directory if it doesn't exist
output_dir = "/dev/shm/evo2_embeddings"
os.makedirs(output_dir, exist_ok=True)

# Iterate through all embeddings and save them as individual .pt files
print(f"Saving {len(evo2_embeddings)} embeddings to {output_dir}...")
for i, (key, value) in tqdm(enumerate(list(evo2_embeddings.items())), desc="Saving embeddings"):
    output_path = os.path.join(output_dir, f"{key}.pt")
    torch.save(value, output_path)

print(f"Successfully saved all embeddings to {output_dir}")

Loading embeddings from /dev/shm/nvidia_evo2_full_embeddings.pkl...
Loaded embeddings for 844 sequences
Sample key: 1SCL_A
Sample value type: <class 'numpy.ndarray'>
Sample embedding shape: (29, 8192)
Saving 844 embeddings to /dev/shm/evo2_embeddings...


Saving embeddings: 844it [00:45, 18.51it/s] 

Successfully saved all embeddings to /dev/shm/evo2_embeddings


: 

In [1]:
from utils import *
val_idx = all_seq_ids("train")

In [2]:
len(val_idx)

449

: 